In [1]:
import subprocess, os, pandas as pd, json, torch
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
import re

# ── STEP 1: Download CodeXGLUE ────────────────────────────────
print("STEP 1: Downloading CodeXGLUE...")
urls = [
    ("train.jsonl", "https://raw.githubusercontent.com/microsoft/CodeXGLUE/main/Code-Code/Defect-detection/dataset/train.jsonl"),
    ("valid.jsonl", "https://raw.githubusercontent.com/microsoft/CodeXGLUE/main/Code-Code/Defect-detection/dataset/valid.jsonl"),
    ("test.jsonl",  "https://raw.githubusercontent.com/microsoft/CodeXGLUE/main/Code-Code/Defect-detection/dataset/test.jsonl"),
]
for fname, url in urls:
    subprocess.run(["wget", "-q", url, "-O", fname])
    print(f"  ✓ {fname} — {os.path.getsize(fname)/1024/1024:.1f} MB")

dfs = []
for fname, _ in urls:
    with open(fname) as f:
        rows = [json.loads(line) for line in f]
    dfs.append(pd.DataFrame(rows))
    print(f"  {fname}: {len(rows)} samples")

df = pd.concat(dfs).reset_index(drop=True)
print(f"\nTotal: {len(df)} samples")
print("Labels:", df['target'].value_counts().to_dict())
df = df.rename(columns={'func': 'func_before', 'target': 'vul'})
print("✓ Download done!")

# ── STEP 2: Clean and balance ─────────────────────────────────
print("\nSTEP 2: Cleaning and balancing...")

def clean_code(code):
    code = re.sub(r'//.*', '', str(code))
    code = re.sub(r'/\*.*?\*/', '', code, flags=re.DOTALL)
    code = re.sub(r'\n\s*\n', '\n', code)
    return code.strip()

df['clean_code'] = df['func_before'].apply(clean_code)

vuln = df[df['vul'] == 1]
safe = df[df['vul'] == 0]
print(f"Vulnerable: {len(vuln)} | Safe: {len(safe)}")

min_count = min(len(vuln), len(safe))
df_bal = pd.concat([
    vuln.sample(n=min_count, random_state=42),
    safe.sample(n=min_count, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Balanced total: {len(df_bal)}")

# ── STEP 3: Split ─────────────────────────────────────────────
print("\nSTEP 3: Splitting...")
train_df, temp_df = train_test_split(df_bal, test_size=0.3,
                    random_state=42, stratify=df_bal['vul'])
val_df, test_df   = train_test_split(temp_df, test_size=0.5,
                    random_state=42, stratify=temp_df['vul'])
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# ── STEP 4: Tokenize (CPU is fine for this) ───────────────────
print("\nSTEP 4: Tokenizing with CodeBERT...")
print("This takes 10-20 mins on CPU. Please wait...")
tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")

def tokenize(texts, label):
    print(f"  Tokenizing {label} ({len(texts)} samples)...")
    return tokenizer(list(texts), max_length=512, padding='max_length',
                     truncation=True, return_tensors='pt')

train_enc = tokenize(train_df['clean_code'], 'train')
val_enc   = tokenize(val_df['clean_code'],   'val')
test_enc  = tokenize(test_df['clean_code'],  'test')
print("✓ Tokenization done!")

# ── STEP 5: Save everything ───────────────────────────────────
print("\nSTEP 5: Saving...")
torch.save({
    'input_ids':      train_enc['input_ids'],
    'attention_mask': train_enc['attention_mask'],
    'labels': torch.tensor(train_df['vul'].values, dtype=torch.long)
}, 'cxg_train.pt')

torch.save({
    'input_ids':      val_enc['input_ids'],
    'attention_mask': val_enc['attention_mask'],
    'labels': torch.tensor(val_df['vul'].values, dtype=torch.long)
}, 'cxg_val.pt')

torch.save({
    'input_ids':      test_enc['input_ids'],
    'attention_mask': test_enc['attention_mask'],
    'labels': torch.tensor(test_df['vul'].values, dtype=torch.long)
}, 'cxg_test.pt')

test_df.to_csv('cxg_test_df.csv', index=False)

print("\nFiles saved:")
for f in ['cxg_train.pt', 'cxg_val.pt', 'cxg_test.pt', 'cxg_test_df.csv']:
    size = os.path.getsize(f) / (1024*1024)
    print(f"  {f} — {size:.1f} MB")

print("\n✓ ALL DONE! Ready for GPU training.")
print("When GPU is free — just run the training cell and it starts instantly!")

STEP 1: Downloading CodeXGLUE...
  ✓ train.jsonl — 0.0 MB
  ✓ valid.jsonl — 0.0 MB
  ✓ test.jsonl — 0.0 MB
  train.jsonl: 0 samples
  valid.jsonl: 0 samples
  test.jsonl: 0 samples

Total: 0 samples


KeyError: 'target'

In [2]:
import json

# Check what's actually in the files
for fname in ["train.jsonl", "valid.jsonl", "test.jsonl"]:
    with open(fname) as f:
        first_row = json.loads(f.readline())
    print(f"{fname} columns: {list(first_row.keys())}")
    print(f"Sample: {first_row}")
    print()

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [5]:
import subprocess, os, pandas as pd

# Install datasets library
subprocess.run(["pip", "install", "datasets", "-q"])

from datasets import load_dataset

print("Downloading CodeXGLUE from HuggingFace...")
ds = load_dataset("google/code_x_glue_cc_defect_detection")
print(ds)

# Convert to pandas
train_df = ds['train'].to_pandas()
val_df   = ds['validation'].to_pandas()
test_df  = ds['test'].to_pandas()

df = pd.concat([train_df, val_df, test_df]).reset_index(drop=True)
print(f"\nTotal: {len(df)} samples")
print("Columns:", df.columns.tolist())
print("Labels:", df['label'].value_counts().to_dict())
print("\nSample:", df['func'][0][:200])

df.to_csv("codexglue_defect.csv", index=False)
print("\n✓ Saved as codexglue_defect.csv")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/17.8M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/2.21M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/2.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/21854 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2732 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2732 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'func', 'target', 'project', 'commit_id'],
        num_rows: 21854
    })
    validation: Dataset({
        features: ['id', 'func', 'target', 'project', 'commit_id'],
        num_rows: 2732
    })
    test: Dataset({
        features: ['id', 'func', 'target', 'project', 'commit_id'],
        num_rows: 2732
    })
})

Total: 27318 samples
Columns: ['id', 'func', 'target', 'project', 'commit_id']


KeyError: 'label'

In [6]:
print("Columns:", df.columns.tolist())
print("\nFirst row:")
print(df.iloc[0])

Columns: ['id', 'func', 'target', 'project', 'commit_id']

First row:
id                                                           0
func         static av_cold int vdadec_init(AVCodecContext ...
target                                                   False
project                                                 FFmpeg
commit_id             973b1a6b9070e2bf17d17568cbaf4043ce931f51
Name: 0, dtype: object


In [7]:
# Fix column names
df = df.rename(columns={'func': 'func_before', 'target': 'vul'})

# Convert True/False to 1/0
df['vul'] = df['vul'].astype(int)

print(f"Total: {len(df)} samples")
print("Labels:", df['vul'].value_counts().to_dict())
print("\nSample code:", df['func_before'][0][:200])

df.to_csv("codexglue_defect.csv", index=False)
print("\n✓ Saved as codexglue_defect.csv!")

Total: 27318 samples
Labels: {0: 14858, 1: 12460}

Sample code: static av_cold int vdadec_init(AVCodecContext *avctx)

{

    VDADecoderContext *ctx = avctx->priv_data;

    struct vda_context *vda_ctx = &ctx->vda_ctx;

    OSStatus status;

    int ret;



    ct

✓ Saved as codexglue_defect.csv!


In [8]:
import torch
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split
import re

print("STEP 2: Cleaning...")
def clean_code(code):
    code = re.sub(r'//.*', '', str(code))
    code = re.sub(r'/\*.*?\*/', '', code, flags=re.DOTALL)
    code = re.sub(r'\n\s*\n', '\n', code)
    return code.strip()

df['clean_code'] = df['func_before'].apply(clean_code)

# Balance 1:1
vuln = df[df['vul'] == 1]
safe = df[df['vul'] == 0]
min_count = min(len(vuln), len(safe))
df_bal = pd.concat([
    vuln.sample(n=min_count, random_state=42),
    safe.sample(n=min_count, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Balanced: {len(df_bal)} samples ({min_count} each)")

# Split
train_df, temp_df = train_test_split(df_bal, test_size=0.3,
                    random_state=42, stratify=df_bal['vul'])
val_df, test_df   = train_test_split(temp_df, test_size=0.5,
                    random_state=42, stratify=temp_df['vul'])
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# Tokenize
print("\nSTEP 3: Tokenizing (10-15 mins on CPU)...")
tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")

def tokenize(texts, name):
    print(f"  Tokenizing {name} ({len(texts)} samples)...")
    return tokenizer(list(texts), max_length=512, padding='max_length',
                     truncation=True, return_tensors='pt')

train_enc = tokenize(train_df['clean_code'], 'train')
val_enc   = tokenize(val_df['clean_code'],   'val')
test_enc  = tokenize(test_df['clean_code'],  'test')

# Save
print("\nSTEP 4: Saving...")
torch.save({'input_ids': train_enc['input_ids'],
            'attention_mask': train_enc['attention_mask'],
            'labels': torch.tensor(train_df['vul'].values, dtype=torch.long)
}, 'cxg_train.pt')

torch.save({'input_ids': val_enc['input_ids'],
            'attention_mask': val_enc['attention_mask'],
            'labels': torch.tensor(val_df['vul'].values, dtype=torch.long)
}, 'cxg_val.pt')

torch.save({'input_ids': test_enc['input_ids'],
            'attention_mask': test_enc['attention_mask'],
            'labels': torch.tensor(test_df['vul'].values, dtype=torch.long)
}, 'cxg_test.pt')

test_df.to_csv('cxg_test_df.csv', index=False)

print("\nFiles saved:")
for f in ['cxg_train.pt', 'cxg_val.pt', 'cxg_test.pt', 'cxg_test_df.csv']:
    size = os.path.getsize(f) / (1024*1024)
    print(f"  {f} — {size:.1f} MB")

print("\n✓ ALL DONE! Ready for GPU training!")

STEP 2: Cleaning...
Balanced: 24920 samples (12460 each)
Train: 17444 | Val: 3738 | Test: 3738

STEP 3: Tokenizing (10-15 mins on CPU)...


config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

  Tokenizing train (17444 samples)...
  Tokenizing val (3738 samples)...
  Tokenizing test (3738 samples)...

STEP 4: Saving...

Files saved:
  cxg_train.pt — 136.4 MB
  cxg_val.pt — 29.2 MB
  cxg_test.pt — 29.2 MB
  cxg_test_df.csv — 13.8 MB

✓ ALL DONE! Ready for GPU training!
